In [2]:
# Arquivo: implementation_prototype.py
# Protótipo de Backend para o SGHSS VidaPlus
# Este código simula a lógica de requisitos críticos: Controle de Acesso (RNF02) e Agendamento (RF03).

import random
from datetime import datetime, timedelta

# --- MOCKUP DE DADOS DO SISTEMA (SIMULAÇÃO DE BANCO DE DADOS) ---

# Usuários mockados com seus perfis (necessário para RNF02 - Controle de Acesso)
MOCK_USERS = {
    "paciente_a": {"senha_hash": "a123", "perfil": "Paciente", "uid": "P001"},
    "dr_silva": {"senha_hash": "d456", "perfil": "Medico", "uid": "M002"},
    "adm_chefe": {"senha_hash": "z999", "perfil": "Administrador", "uid": "A003"},
}

# Agenda mockada para verificar conflitos de horário (necessário para RF03)
# Formato: { 'MedicoID': [ (data_hora_inicio, data_hora_fim), ... ] }
MOCK_AGENDA = {
    "M002": [
        (datetime(2025, 12, 10, 10, 0), datetime(2025, 12, 10, 11, 0)),
        (datetime(2025, 12, 10, 14, 0), datetime(2025, 12, 10, 15, 0)),
    ]
}

# --- FUNÇÕES DE LÓGICA DE NEGÓCIO ---

def login_usuario(usuario, senha):
    """
    RNF02: Realiza o login e verifica o perfil do usuário para controle de acesso.

    Args:
        usuario (str): Nome de usuário fornecido.
        senha (str): Senha fornecida (simulando hash simples 'senha_hash').

    Returns:
        tuple: (bool, str, str). Indica (sucesso, perfil, uid do usuário).
    """
    print(f"Tentativa de login para: {usuario}")
    if usuario in MOCK_USERS:
        usuario_data = MOCK_USERS[usuario]
        # Simula a verificação do hash da senha
        if usuario_data["senha_hash"] == senha:
            print(f"Sucesso: Usuário '{usuario}' logado com perfil '{usuario_data['perfil']}'.")
            # Este resultado é usado nos Testes de Controle de Acesso (Seção 6)
            return True, usuario_data["perfil"], usuario_data["uid"]
        else:
            print("Falha: Senha inválida.")
            return False, "Nenhum", None
    else:
        print("Falha: Usuário não encontrado.")
        return False, "Nenhum", None

def agendar_consulta(id_medico, data_hora_inicio, duracao_minutos=60):
    """
    RF03: Tenta agendar uma consulta, verificando a disponibilidade do médico.

    Args:
        id_medico (str): ID do profissional de saúde.
        data_hora_inicio (datetime): Data e hora desejadas para a consulta.
        duracao_minutos (int): Duração da consulta em minutos (padrão 60).

    Returns:
        bool: True se o agendamento foi bem-sucedido, False caso contrário.
    """
    print(f"\nTentativa de agendamento para o Médico {id_medico} às {data_hora_inicio.strftime('%Y-%m-%d %H:%M')}")

    data_hora_fim = data_hora_inicio + timedelta(minutes=duracao_minutos)

    # 1. Verifica se o médico existe no sistema de agenda
    if id_medico not in MOCK_AGENDA:
        # Se o médico não tiver agendamentos, ele está livre
        MOCK_AGENDA[id_medico] = []

    agenda_do_medico = MOCK_AGENDA[id_medico]

    # 2. Verifica a Regra de Negócio: Conflito de Horário
    for inicio_existente, fim_existente in agenda_do_medico:
        # Se o novo agendamento começar antes do fim de um existente E
        # terminar depois do início desse existente -> Conflito
        if (data_hora_inicio < fim_existente) and (data_hora_fim > inicio_existente):
            print("Falha de Negócio: Horário em conflito com uma consulta existente.")
            # Este é o cenário de falha para o CTF01 (Seção 6)
            return False

    # 3. Se não há conflito, o agendamento é realizado
    MOCK_AGENDA[id_medico].append((data_hora_inicio, data_hora_fim))
    MOCK_AGENDA[id_medico].sort(key=lambda x: x[0]) # Ordena para manter a agenda organizada
    print("Sucesso: Consulta agendada e registrada no sistema.")
    # Este é o cenário de sucesso para o CTF01 (Seção 6)
    return True

# --- EXEMPLOS DE EXECUÇÃO (Para validação rápida) ---

if __name__ == "__main__":
    print("--- Teste de Login (RNF02) ---")

    # Sucesso - Administrador
    sucesso_adm, perfil_adm, uid_adm = login_usuario("adm_chefe", "z999")

    # Falha - Senha inválida (Cenário de Teste de Segurança)
    login_usuario("paciente_a", "senha_errada")

    print("\n--- Teste de Agendamento (RF03) ---")

    # 1. Agendamento em horário livre (Sucesso)
    hora_livre = datetime(2025, 12, 10, 11, 30) # Entre 11:00 e 14:00
    agendar_consulta("M002", hora_livre)

    # 2. Agendamento em horário em conflito (Falha - CTF01)
    hora_conflito = datetime(2025, 12, 10, 10, 45) # Conflito com a primeira consulta
    agendar_consulta("M002", hora_conflito)

    # 3. Agendamento em um novo dia (Sucesso)
    hora_novo_dia = datetime(2025, 12, 11, 9, 0)
    agendar_consulta("M002", hora_novo_dia)

    print("\n--- Agenda Final do Dr. Silva (M002) ---")
    for inicio, fim in MOCK_AGENDA.get("M002", []):
        print(f" - Início: {inicio.strftime('%H:%M')} | Fim: {fim.strftime('%H:%M')}")

--- Teste de Login (RNF02) ---
Tentativa de login para: adm_chefe
Sucesso: Usuário 'adm_chefe' logado com perfil 'Administrador'.
Tentativa de login para: paciente_a
Falha: Senha inválida.

--- Teste de Agendamento (RF03) ---

Tentativa de agendamento para o Médico M002 às 2025-12-10 11:30
Sucesso: Consulta agendada e registrada no sistema.

Tentativa de agendamento para o Médico M002 às 2025-12-10 10:45
Falha de Negócio: Horário em conflito com uma consulta existente.

Tentativa de agendamento para o Médico M002 às 2025-12-11 09:00
Sucesso: Consulta agendada e registrada no sistema.

--- Agenda Final do Dr. Silva (M002) ---
 - Início: 10:00 | Fim: 11:00
 - Início: 11:30 | Fim: 12:30
 - Início: 14:00 | Fim: 15:00
 - Início: 09:00 | Fim: 10:00
